# End-to-End Opinf for multiple initial conditions
## What this notebook can do
- Compute a POD basis for 1+ intial conditions
- Find the operator(s) for $\gamma_n$, $\gamma_c$, and state
- Use learned operator(s) and POD to compute predictions across unseen data

## Memory Management Notes

**Memory-mapped files created in `output_path`:**
- `memmap_Q_train.dat` - Full training snapshots (only during step_1)
- `memmap_Q_test.dat` - Full test snapshots (only during step_1)  
- `memmap_X_recon_full.dat` - Full-space model reconstruction
- `memmap_X_truth_full.dat` - Full-space ground truth reconstruction

**Use cases:**
- **Full run** (`step_1=True, step_2=True`): Computes POD, trains ROM, makes predictions
- **ROM only** (`step_1=False, step_2=True`): Loads existing POD, trains new ROM
- **Predictions only** (`step_1=False, step_2=False`): Loads existing POD and ROM

**Cleanup:** To free disk space after you're done, run:
```python
for name in ["Q_train", "Q_test", "X_recon_full", "X_truth_full"]:
    cleanup_memmap(name)
```

In [1]:
### New best version
# pip install -e . --no-deps  # Install without dependencies first
# pip install -e .            # Then install with pinned dependencies

In [2]:
%matplotlib inline
from opinf_for_hw.data_proc import *
from opinf_for_hw.postproc import *
from opinf_for_hw.utils.helpers import loader
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import h5py
from IPython import display
import xarray as xr
import time
import gc
import os

# General Config
cluster = True
show_animations = False
show_plots = True
ENGINE = "h5netcdf"
plt.rcParams['animation.embed_limit'] = 250
plt.rcParams["image.cmap"] = "bwr" # Other options: seismic

# Control Config
step_1 = False # Compute POD
step_2 = False # Compute ROM

# Memory Config
use_chunked_projection = False  # Set True to project in chunks (lower peak memory)


def bprint(string: str):
    print("\033[1m" + string + "\033[0m")

def get_memmap_path(name):
    """Get path for memory-mapped file."""
    return os.path.join(output_path, f"memmap_{name}.dat")

def cleanup_memmap(name):
    """Remove memory-mapped file if it exists."""
    path = get_memmap_path(name)
    if os.path.exists(path):
        os.remove(path)

if cluster:
    bprint("Using cluster settings")
    from config.cluster import *
else:
    bprint("Using local settings")
    from config.local import *

if r > svd_save:
    bprint("Warning! r value larger than svd_save")

bprint(f"Number of training trajectories/files: {len(training_files)}")
bprint(f"Number of test trajectories/files: {len(test_files)}")

Using cluster settings
Number of training trajectories/files: 1
Number of test trajectories/files: 1


In [3]:
np.__config__.show()

blas_mkl_info:
    libraries = ['mkl_rt', 'pthread']
    library_dirs = ['/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/lib/intel64']
    define_macros = [('SCIPY_MKL_H', None), ('HAVE_CBLAS', None)]
    include_dirs = ['/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl', '/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/include', '/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/lib']
blas_opt_info:
    libraries = ['mkl_rt', 'pthread']
    library_dirs = ['/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/lib/intel64']
    define_macros = [('SCIPY_MKL_H', None), ('HAVE_CBLAS', None)]
    include_dirs = ['/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl', '/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/include', '/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/lib']
lapack_mkl_info:
    libraries = ['mkl_rt', 'pthread']
    library_dirs = ['/opt/intel/compilers_and_libraries_2019.5.281/linux/mkl/lib/intel64']
    define_macro

## Step 1: Compute POD basis

### Step 1.1: Load training and test data

In [4]:
def load_and_process_snapshots(file_path, i, dataset_name=""):
    """Helper function to load and process a single snapshot file.
    Returns processed array and its shape info (doesn't append to list).
    """
    print(f"  Loading IC {i+1}: {file_path}")
    fh = xr.open_dataset(file_path, 
                         engine=ENGINE, 
                         phony_dims="sort"
                        )
    
    # Get density and phi as numpy arrays
    density = fh["density"].values
    phi = fh["phi"].values
    fh.close()  # Explicitly close file handle
    
    if i == 0 and show_plots:
        plt.imshow(density[0])
        plt.colorbar()
        plt.show()
        bprint(f"{dataset_name}IC{i} shape: {density.shape}")
    
    # If already 3D (time, y, x), use as-is. If 2D (time, flattened), reshape to 3D
    if density.ndim == 2:
        n_time = density.shape[0]
        grid_size = int(np.sqrt(density.shape[1]))
        density = density.reshape(n_time, grid_size, grid_size)
        phi = phi.reshape(n_time, grid_size, grid_size)
    
    # Stack fields: (time, y, x) -> (field, time, y, x)
    Q_ic = np.stack([density, phi], axis=0)  # Shape: (2, time, y, x)
    del density, phi  # Free memory immediately
    
    # Transpose to (field, y, x, time) before flattening
    Q_ic = Q_ic.transpose(0, 2, 3, 1)  # Shape: (2, 256, 256, time)
    
    # Now reshape: flatten spatial dimensions
    n_field, n_y, n_x, n_time = Q_ic.shape
    Q_ic = Q_ic.reshape(n_field * n_y * n_x, n_time)  # Shape: (131072, time)
    
    print(f"    Shape: {Q_ic.shape}")
    return Q_ic


if step_1:
    bprint("Reading snapshot(s) - Building memory-mapped arrays")
    
    # First pass: determine total sizes
    train_timesteps = []
    test_timesteps = []
    n_spatial = None
    
    for file_path in training_files:
        with xr.open_dataset(file_path, engine=ENGINE, phony_dims="sort") as fh:
            n_time = fh["density"].shape[0]
            if n_spatial is None:
                if fh["density"].ndim == 3:
                    n_spatial = 2 * fh["density"].shape[1] * fh["density"].shape[2]
                else:
                    n_spatial = 2 * fh["density"].shape[1]
            train_timesteps.append(n_time)
    
    for file_path in test_files:
        with xr.open_dataset(file_path, engine=ENGINE, phony_dims="sort") as fh:
            test_timesteps.append(fh["density"].shape[0])
    
    total_train_time = sum(train_timesteps)
    total_test_time = sum(test_timesteps)
    
    bprint(f"Creating memory-mapped arrays: {n_spatial} spatial × {total_train_time} train timesteps")
    
    # Create memory-mapped arrays for Q_train and Q_test
    cleanup_memmap("Q_train")
    cleanup_memmap("Q_test")
    
    Q_train = np.memmap(get_memmap_path("Q_train"), dtype='float64', mode='w+', 
                        shape=(n_spatial, total_train_time))
    Q_test = np.memmap(get_memmap_path("Q_test"), dtype='float64', mode='w+', 
                       shape=(n_spatial, total_test_time))
    
    # Store timestep boundaries for later IC extraction
    train_boundaries = [0] + list(np.cumsum(train_timesteps))
    test_boundaries = [0] + list(np.cumsum(test_timesteps))
    
    # Load training data directly into memmap
    for i, file_path in enumerate(training_files):
        Q_ic = load_and_process_snapshots(file_path, i, dataset_name="")
        start_idx = train_boundaries[i]
        end_idx = train_boundaries[i + 1]
        Q_train[:, start_idx:end_idx] = Q_ic
        
        if i == 0 and show_plots:
            img = Q_ic[:256*256, 0].reshape(256, 256)
            plt.imshow(img)
            plt.colorbar()
            plt.show()
        del Q_ic
        gc.collect()
    
    bprint(f"Combined training data shape: {Q_train.shape}")
    
    # Load test data directly into memmap
    for i, file_path in enumerate(test_files):
        Q_ic = load_and_process_snapshots(file_path, i, dataset_name="Test ")
        start_idx = test_boundaries[i]
        end_idx = test_boundaries[i + 1]
        Q_test[:, start_idx:end_idx] = Q_ic
        del Q_ic
        gc.collect()
    
    bprint(f"Combined test data shape: {Q_test.shape}")
    
    # Save boundaries for IC extraction
    np.savez(output_path + "data_boundaries.npz", 
             train_boundaries=train_boundaries, 
             test_boundaries=test_boundaries,
             n_spatial=n_spatial)
    
else:
    bprint("Skipping data loading (step_1=False, will load POD directly)")

Skipping data loading (step_1=False, will load POD directly)


### Step 1.2: Compute POD

In [5]:
if step_1: 
    # Compute POD basis from combined training data
    bprint("Computing POD basis from all training trajectories...")
    start_time = time.time()

    U, S, _ = np.linalg.svd(Q_train, full_matrices=False)

    elapsed = time.time() - start_time
    print(f"  POD computation completed in {elapsed:.3f} seconds.")
    
    # Save POD data
    POD_file_multi = output_path + "POD_multi_IC.npz"
    np.savez(POD_file_multi, S=S, U=U)
    print(f"  Saved POD basis to {POD_file_multi}")
    print(f"  U shape: {U.shape}, S shape: {S.shape}")
    
    gc.collect()

else:
    # Load previous POD basis
    bprint("Loading POD basis...")
    POD_file_multi = output_path + "POD_multi_IC.npz"
    POD_multi = np.load(POD_file_multi)
    S, U = POD_multi['S'], POD_multi['U']
    del POD_multi
    gc.collect()
    print(f"  Loaded POD basis from {POD_file_multi}")
    print(f"  U shape: {U.shape}, S shape: {S.shape}")

Loading POD basis...
  Loaded POD basis from /work2/10407/anthony50102/frontera/data/sciml_roms_hasegawa_wakatani/POD_multi_IC.npz
  U shape: (131072, 16001), S shape: (16001,)


### Step 1.3: Project train & test

In [6]:
if step_1:
    # Project training data
    bprint("Projecting training data...")
    # Use only the modes we need (truncate U to r modes for projection storage)
    Ur = U[:, :r]
    
    Xhat_train = Q_train.T @ Ur  # Shape: (n_time, r) - much smaller!
    Xhat_train_file = output_path + "X_hat_train_multi_IC.npy"
    np.save(Xhat_train_file, Xhat_train)
    print(f"  Saved to {Xhat_train_file}, shape: {Xhat_train.shape}")
    
    # Project test data
    bprint("Projecting test data...")
    Xhat_test = Q_test.T @ Ur
    Xhat_test_file = output_path + "X_hat_test_multi_IC.npy"
    np.save(Xhat_test_file, Xhat_test)
    print(f"  Saved to {Xhat_test_file}, shape: {Xhat_test.shape}")
    
    del Ur
    gc.collect()

else:
    # Load pre-computed projections
    bprint("Loading pre-computed projections...")
    Xhat_train_file = output_path + "X_hat_train_multi_IC.npy"
    Xhat_test_file = output_path + "X_hat_test_multi_IC.npy"
    
    Xhat_train = np.load(Xhat_train_file)
    Xhat_test = np.load(Xhat_test_file)
    print(f"  Loaded Xhat_train: {Xhat_train.shape}")
    print(f"  Loaded Xhat_test: {Xhat_test.shape}")

Loading pre-computed projections...
  Loaded Xhat_train: (16001, 100)
  Loaded Xhat_test: (16001, 100)


In [7]:
if step_1:
    # Save initial conditions for later use
    bprint("Saving initial conditions...")
    
    # Load boundaries
    boundaries = np.load(output_path + "data_boundaries.npz")
    train_boundaries = boundaries['train_boundaries']
    test_boundaries = boundaries['test_boundaries']
    
    # Extract ICs from memmap (only first timestep of each trajectory)
    train_ICs = np.array([Q_train[:, train_boundaries[i]] for i in range(len(training_files))])
    test_ICs = np.array([Q_test[:, test_boundaries[i]] for i in range(len(test_files))])
    
    # Reduced ICs from projected data
    train_ICs_reduced = np.array([Xhat_train[train_boundaries[i], :] for i in range(len(training_files))])
    test_ICs_reduced = np.array([Xhat_test[test_boundaries[i], :] for i in range(len(test_files))])
    
    np.savez(
        output_path + "initial_conditions_multi_IC.npz",
        train_ICs=train_ICs,
        test_ICs=test_ICs,
        train_ICs_reduced=train_ICs_reduced,
        test_ICs_reduced=test_ICs_reduced
    )
    print(f"  Saved ICs: train_ICs {train_ICs.shape}, test_ICs {test_ICs.shape}")
    
    del train_ICs, test_ICs, train_ICs_reduced, test_ICs_reduced
    
    # Clean up large memory-mapped arrays - we're done with them
    bprint("Cleaning up large arrays...")
    del Q_train, Q_test
    gc.collect()
    
    # Optionally remove memmap files to free disk space
    # cleanup_memmap("Q_train")
    # cleanup_memmap("Q_test")
    
    bprint("Done with Step 1.")

else:
    bprint("Skipping IC saving (step_1=False)")

Skipping IC saving (step_1=False)


## Step 2: Compute ROM

In [8]:
def solve_opinf_difference_model(s0, n_steps, f):
    s = np.zeros((np.size(s0), n_steps))
    is_nan = False

    s[:, 0] = s0
    for i in range(n_steps - 1):
        s[:, i + 1] = f(s[:, i])

        if np.any(np.isnan(s[:, i + 1])):
            print("NaN encountered at iteration " + str(i + 1))
            is_nan = True
            break

    return is_nan, s

### Step 2.1: Prep data

In [9]:
bprint("Prepare the data for learning...")

# Ensure we have Xhat_train loaded
if 'Xhat_train' not in dir() or Xhat_train is None:
    bprint("Loading Xhat_train from disk...")
    Xhat_train = np.load(output_path + "X_hat_train_multi_IC.npy")

print(f"Training data shape: {Xhat_train.shape}")

# Truncate to r modes if needed (in case we loaded full projection)
if Xhat_train.shape[1] > r:
    Xhat_train = Xhat_train[:, :r]
    print(f"  Truncated to r={r} modes: {Xhat_train.shape}")

# Prepare state evolution data (views, not copies)
X_state = Xhat_train[:-1, :]
Y_state = Xhat_train[1:, :]

s = int(r * (r + 1) / 2)
d_state = r + s
d_out = r + s + 1

X_state2 = get_x_sq(X_state)
D_state = np.concatenate((X_state, X_state2), axis=1)
D_state_2 = D_state.T @ D_state
bprint("State learning data prepared")

Prepare the data for learning...
Training data shape: (16001, 100)
State learning data prepared


In [10]:
bprint("Prepare the output learning data")

X_out = Xhat_train
K = X_out.shape[0]
E = np.ones((K, 1))

mean_Xhat = np.mean(X_out, axis=0)
Xhat_out = X_out - mean_Xhat[np.newaxis, :]

local_min = np.min(X_out)
local_max = np.max(X_out)
local_scaling = np.maximum(np.abs(local_min), np.abs(local_max))
scaling_Xhat = local_scaling

Xhat_out /= scaling_Xhat
Xhat_out2 = get_x_sq(Xhat_out)

D_out = np.concatenate((Xhat_out, Xhat_out2, E), axis=1)
D_out_2 = D_out.T @ D_out

print(f"D_out shape: {D_out.shape}")
print(f"D_out_2 condition number: {np.linalg.cond(D_out_2):.2e}")
bprint("Done")

Prepare the output learning data
D_out shape: (16001, 5151)
D_out_2 condition number: 2.39e+17
Done


In [11]:
bprint("Load derived quantities from all training trajectories")

Gamma_n_list = []
Gamma_c_list = []

for file_path in training_files:
    fh = loader(file_path, ENGINE=ENGINE)
    Gamma_n_list.append(fh["gamma_n"].data)
    Gamma_c_list.append(fh["gamma_c"].data)

# Concatenate all trajectories
Gamma_n = np.concatenate(Gamma_n_list)
Gamma_c = np.concatenate(Gamma_c_list)

mean_Gamma_n_ref = np.mean(Gamma_n)
std_Gamma_n_ref = np.std(Gamma_n, ddof=1)

mean_Gamma_c_ref = np.mean(Gamma_c)
std_Gamma_c_ref = np.std(Gamma_c, ddof=1)

Y_Gamma = np.vstack((Gamma_n, Gamma_c))

print(f"Gamma_n shape: {Gamma_n.shape}")
print(f"Gamma_c shape: {Gamma_c.shape}")
print(f"Y_Gamma shape: {Y_Gamma.shape}")
print(f"X_out shape (for output learning): {X_out.shape}")
print(f"Shape compatibility check: Y_Gamma cols ({Y_Gamma.shape[1]}) vs X_out rows ({X_out.shape[0]})")

if Y_Gamma.shape[1] != X_out.shape[0]:
    raise ValueError(f"Shape mismatch: Y_Gamma has {Y_Gamma.shape[1]} columns but X_out has {X_out.shape[0]} rows")

print(f"Mean Gamma_n: {mean_Gamma_n_ref:.4f}, Std: {std_Gamma_n_ref:.4f}")
print(f"Mean Gamma_c: {mean_Gamma_c_ref:.4f}, Std: {std_Gamma_c_ref:.4f}")
print("Done")

Load derived quantities from all training trajectories
 ERROR: Could not open file /work2/10407/anthony50102/frontera/data/hw2d_sim/t600_d256x256_raw/hw2d_sim_step0.025_end1_pts512_c11_k015_N3_nu5e-8_20250315142044_11702_0.h5: variable '/density' has no dimension scale associated with axis 0. 
Use phony_dims='sort' for sorted naming or phony_dims='access' for per access naming. 
  Retrying with phony_dims='sort'...
Gamma_n shape: (16001,)
Gamma_c shape: (16001,)
Y_Gamma shape: (2, 16001)
X_out shape (for output learning): (16001, 100)
Shape compatibility check: Y_Gamma cols (16001) vs X_out rows (16001)
Mean Gamma_n: 0.5901, Std: 0.0416
Mean Gamma_c: 0.5843, Std: 0.0354
Done


### Step 2.2: Model Regularization Sweep

In [12]:
if step_2:
    bprint("BEGIN PARAMETER SWEEP - TRACKING BEST L2 ERROR")
    
    best_l2_error = np.inf
    best_model = None
    best_alphas = None
    
    for alpha_state_lin in ridge_alf_lin_all:
        for alpha_state_quad in ridge_alf_quad_all:
            print("alpha_lin = %.2E" % alpha_state_lin)
            print("alpha_quad = %.2E" % alpha_state_quad)
    
            regg = np.zeros(d_state)
            regg[:r] = alpha_state_lin
            regg[r : r + s] = alpha_state_quad
            regularizer = np.diag(regg)
            D_state_reg = D_state_2 + regularizer
    
            O = np.linalg.solve(D_state_reg, np.dot(D_state.T, Y_state)).T
    
            A = O[:, :r]
            F = O[:, r : r + s]
    
            # Compute training predictions
            Y_state_pred = A @ X_state.T + F @ X_state2.T
            Y_state_pred = Y_state_pred.T
    
            # Check for NaNs
            if np.any(np.isnan(Y_state_pred)) or np.any(np.isinf(Y_state_pred)):
                print("  Skipping due to NaN/Inf in training predictions")
                continue
    
            # Compute L2 error on training data
            l2_error = np.linalg.norm(Y_state - Y_state_pred, 'fro')
            relative_l2_error = l2_error / np.linalg.norm(Y_state, 'fro')
    
            print(f"  L2 error: {l2_error:.6e} | Relative L2: {relative_l2_error:.6e}")
    
            # If this is the best model so far, save it
            if l2_error < best_l2_error:
                best_l2_error = l2_error
                best_model = {
                    'A': A.copy(),
                    'F': F.copy(),
                    # 'C': C.copy(),
                    # 'G': G.copy(),
                    # 'c': c.copy(),
                    'relative_l2_error': relative_l2_error,
                    'absolute_l2_error': l2_error
                }
                best_alphas = {
                    'alpha_state_lin': alpha_state_lin,
                    'alpha_state_quad': alpha_state_quad,
                    # 'alpha_out_lin': alpha_out_lin,
                    # 'alpha_out_quad': alpha_out_quad
                }
                print(f"  ✓ New best model! L2 error: {l2_error:.6e}")
    
    bprint("Done")
else:
    bprint("Skipping model search...")

Skipping model search...


In [13]:
if step_2:
    bprint("Using computed best model...")
    if best_model is not None:
        print(f"Best L2 error: {best_model['absolute_l2_error']:.6e}")
        print(f"Best relative L2 error: {best_model['relative_l2_error']:.6e}")
        print(f"Best regularization parameters:")
        print(f"  alpha_state_lin:  {best_alphas['alpha_state_lin']:.2e}")
        print(f"  alpha_state_quad: {best_alphas['alpha_state_quad']:.2e}")
        
        # Save the best model
        np.savez(
            output_path + "best_model_l2_multi_IC_r" + str(r) + ".npz",
            A=best_model['A'],
            F=best_model['F'],
            alpha_state_lin=best_alphas['alpha_state_lin'],
            alpha_state_quad=best_alphas['alpha_state_quad'],
            l2_error=best_model['absolute_l2_error'],
            relative_l2_error=best_model['relative_l2_error']
        )
        bprint(f"Best model saved to {output_path}best_model_l2_multi_IC_r{r}.npz")
    else:
        bprint("WARNING: No valid models found during sweep!")
else:
    bprint("Loading pre-computed best model...")
    model_file = output_path + "best_model_l2_multi_IC_r" + str(r) + ".npz"
    best_model_data = np.load(model_file)
    best_model = {
        'A': best_model_data['A'],
        'F': best_model_data['F'],
        'absolute_l2_error': float(best_model_data['l2_error']),
        'relative_l2_error': float(best_model_data['relative_l2_error'])
    }
    best_alphas = {
        'alpha_state_lin': float(best_model_data['alpha_state_lin']),
        'alpha_state_quad': float(best_model_data['alpha_state_quad'])
    }
    print(f"  Loaded model from {model_file}")
    print(f"  L2 error: {best_model['absolute_l2_error']:.6e}")

Loading pre-computed best model...
  Loaded model from /work2/10407/anthony50102/frontera/data/sciml_roms_hasegawa_wakatani/best_model_l2_multi_IC_r100.npz
  L2 error: 3.090425e-02


## Step 3: Make Predictions with model

In [14]:
if best_model is not None:
    # Save predictions from the best model
    bprint("Computing predictions in reduced space...")
    
    # Ensure X_state and X_state2 are available
    if 'X_state' not in dir():
        Xhat_train = np.load(output_path + "X_hat_train_multi_IC.npy")[:, :r]
        X_state = Xhat_train[:-1, :]
        X_state2 = get_x_sq(X_state)
    
    Y_state_pred = best_model['A'] @ X_state.T + best_model['F'] @ X_state2.T
    Y_state_pred = Y_state_pred.T
    np.savez(output_path + "Y_state_best_model_l2_multi_IC_r" + str(r) + ".npz", Y_state_pred=Y_state_pred)
    
    ###########################
    bprint("Reconstruct training data in original space")
    
    # Load POD basis if not in memory
    if 'U' not in dir():
        bprint("  Loading POD basis...")
        POD_multi = np.load(output_path + "POD_multi_IC.npz")
        U = POD_multi['U']
        del POD_multi
    
    Vr = U[:, :r]  # Truncated basis
    
    # Reconstruct state trajectory in POD space (one-step predictions)
    A_best = best_model['A']
    F_best = best_model['F']
    
    Y_state_pred = A_best @ X_state.T + F_best @ X_state2.T
    X_recon_pod = Y_state_pred.T  # Shape: (n_timesteps-1, r)
    
    # Add the initial condition to get full trajectory
    X_full_pod = np.vstack([X_state[0, :], X_recon_pod])  # Shape: (n_timesteps, r)
    
    # Project back to full space using memory-mapped output
    bprint("  Projecting to full space (memory-mapped)...")
    n_timesteps = X_full_pod.shape[0]
    n_spatial = Vr.shape[0]
    
    cleanup_memmap("X_recon_full")
    X_full_space = np.memmap(get_memmap_path("X_recon_full"), dtype='float64', mode='w+',
                             shape=(n_timesteps, n_spatial))
    
    # Project - chunked or all at once based on config
    if use_chunked_projection:
        bprint("  Using chunked projection...")
        chunk_size = 1000
        for start in range(0, n_timesteps, chunk_size):
            end = min(start + chunk_size, n_timesteps)
            X_full_space[start:end, :] = X_full_pod[start:end, :] @ Vr.T
    else:
        X_full_space[:] = X_full_pod @ Vr.T
    
    print(f"Reconstructed data shape: {X_full_space.shape}")
    print(f"  (represents {n_timesteps} timesteps × {n_spatial//2} spatial points × 2 fields)")
    
    ###########################
    bprint("Create ground truth POD reconstruction for comparison")
    
    # Load only first training file for ground truth comparison
    with h5py.File(training_files[0], 'r') as f:
        density = f['density'][:]
        pot = f['phi'][:]
        # Reshape to match our flattening convention
        n_time_truth = density.shape[0]
        if density.ndim == 3:
            density = density.transpose(1, 2, 0).reshape(-1, n_time_truth)
            pot = pot.transpose(1, 2, 0).reshape(-1, n_time_truth)
        X_train_original = np.vstack([density, pot]).T  # Shape: (n_time, n_spatial)
        del density, pot
    
    # Project to POD space and back
    X_truth_pod = X_train_original @ Vr  # Project down: (n_timesteps, r)
    
    cleanup_memmap("X_truth_full")
    X_truth_recon = np.memmap(get_memmap_path("X_truth_full"), dtype='float64', mode='w+',
                              shape=X_train_original.shape)
    
    # Project back - chunked or all at once
    if use_chunked_projection:
        chunk_size = 1000
        for start in range(0, X_train_original.shape[0], chunk_size):
            end = min(start + chunk_size, X_train_original.shape[0])
            X_truth_recon[start:end, :] = X_truth_pod[start:end, :] @ Vr.T
    else:
        X_truth_recon[:] = X_truth_pod @ Vr.T
    
    del X_train_original
    gc.collect()
    
    print(f"Ground truth POD reconstruction shape: {X_truth_recon.shape}")
    
    # Save POD coefficients (small) and metadata
    np.savez(
        output_path + "training_reconstruction_multi_IC_r" + str(r) + ".npz",
        X_recon_pod=X_full_pod,         # POD coefficient trajectory (predictions)
        X_truth_pod=X_truth_pod,        # POD coefficient trajectory (ground truth)
        n_timesteps=n_timesteps,
        n_spatial=n_spatial//2,
        n_fields=2,
        # Store paths to memmap files for full reconstructions
        X_recon_full_memmap=get_memmap_path("X_recon_full"),
        X_truth_full_memmap=get_memmap_path("X_truth_full")
    )
    bprint("Training reconstruction metadata saved!")
    bprint(f"Full reconstructions stored in memory-mapped files at {output_path}")

else:
    bprint("WARNING: No valid models found!")

Computing predictions in reduced space...
Reconstruct training data in original space
  Projecting to full space (memory-mapped)...
Reconstructed data shape: (16001, 131072)
  (represents 16001 timesteps × 65536 spatial points × 2 fields)
Create ground truth POD reconstruction for comparison
Ground truth POD reconstruction shape: (16001, 131072)
Training reconstruction metadata saved!
Full reconstructions stored in memory-mapped files at /work2/10407/anthony50102/frontera/data/sciml_roms_hasegawa_wakatani/


In [15]:
### Load the prediction and ground truth
bprint("Loading reconstruction data...")

pred_file = np.load(output_path + "training_reconstruction_multi_IC_r" + str(r) + ".npz", allow_pickle=True)
print(f"Keys: {list(pred_file.keys())}")

n_timesteps = int(pred_file['n_timesteps'])
n_spatial = int(pred_file['n_spatial'])
n_fields = int(pred_file['n_fields'])

# Load memory-mapped full reconstructions
X_recon = np.memmap(get_memmap_path("X_recon_full"), dtype='float64', mode='r',
                    shape=(n_timesteps, n_spatial * n_fields))
X_truth = np.memmap(get_memmap_path("X_truth_full"), dtype='float64', mode='r',
                    shape=(n_timesteps, n_spatial * n_fields))

print(f"Reconstruction shape: {X_recon.shape}")
print(f"Ground truth shape: {X_truth.shape}")
print(f"Memory-mapped (not loaded into RAM until accessed)")

Loading reconstruction data...
Keys: ['X_recon_pod', 'X_truth_pod', 'n_timesteps', 'n_spatial', 'n_fields', 'X_recon_full_memmap', 'X_truth_full_memmap']
Reconstruction shape: (16001, 131072)
Ground truth shape: (16001, 131072)
Memory-mapped (not loaded into RAM until accessed)


In [16]:
if show_animations:
    # Create 2x2 subplot grid
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    ax = axes.flatten()
    
    # Downsample - only load the frames we need from memmap
    subsample = 100
    frame_indices = list(range(0, n_timesteps, subsample))
    n_frames = len(frame_indices)
    
    # Load only subsampled frames into memory
    bprint(f"Loading {n_frames} subsampled frames for animation...")
    X_recon_subbed = np.array([X_recon[i].reshape(2, 256, 256) for i in frame_indices])
    X_truth_subbed = np.array([X_truth[i].reshape(2, 256, 256) for i in frame_indices])
    
    # Initialize images
    # Top row: Predictions
    im0 = ax[0].imshow(X_recon_subbed[0, 0], animated=True)
    ax[0].set_title('Prediction - Density')
    ax[0].axis('off')
    
    im1 = ax[1].imshow(X_recon_subbed[0, 1], animated=True)
    ax[1].set_title('Prediction - Phi')
    ax[1].axis('off')
    
    # Bottom row: Ground Truth
    im2 = ax[2].imshow(X_truth_subbed[0, 0], animated=True)
    ax[2].set_title('Ground Truth - Density')
    ax[2].axis('off')
    
    im3 = ax[3].imshow(X_truth_subbed[0, 1], animated=True)
    ax[3].set_title('Ground Truth - Phi')
    ax[3].axis('off')
    
    plt.tight_layout()
    
    # Define animate as a closure that captures the data
    def animate(frame):
        im0.set_data(X_recon_subbed[frame, 0])
        im1.set_data(X_recon_subbed[frame, 1])
        im2.set_data(X_truth_subbed[frame, 0])
        im3.set_data(X_truth_subbed[frame, 1])
        return (im0, im1, im2, im3)
    
    # Create animation BEFORE any cleanup
    ani = animation.FuncAnimation(
        fig, animate,
        frames=n_frames,
        interval=50,
        blit=False  # Keep blit=False
    )
    
    # Save/display immediately while data is still in scope
    ani.save(output_path + "training_reconstruction_multi_IC_r" + str(r) + ".gif", writer="pillow")
    print(f"Saved the animation to: {output_path}training_reconstruction_multi_IC_r{r}.gif")
    
    # NOW display (after saving)
    display.display(display.HTML(ani.to_jshtml()))
    
    # Close the figure to prevent matplotlib from trying to redraw it
    plt.close(fig)
    
    # Clean up animation arrays AFTER everything is done
    del X_recon_subbed, X_truth_subbed, ani
    gc.collect()
    
else:
    print("Not showing animations")

Not showing animations
